# Constrained line-number plus phase decomposition

This notebook tests the hypothesis

\[h_{i,t} \approx \mu + v k_{i,t} + W z(u_{i,t}),\]

where `k` is line number, `u` is normalized within-line phase, `v` is one line-number direction, and the phase component is constrained to lie in the orthogonal complement of `v`.

The constrained model is compared with an unconstrained additive model and a model with separate line-number offsets. Evaluation is performed in full activation space on held-out examples.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import SplineTransformer

import project_main.checkpoints as project_checkpoints
import project_main.data as project_data
import project_main.model as project_model
import project_main.tokens as project_tokens

RUN_DIR = Path("../results/line_breaks_results")
CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "final.pt"
N_TRAIN_EXAMPLES = 2_000
N_EVAL_EXAMPLES = 500
COLLECTION_BATCH_SIZE = 128
TRAIN_SEED = 26_001
EVAL_SEED = 26_002
N_SPLINE_KNOTS = 12
RIDGE_ALPHA = 1.0
N_ALTERNATING_STEPS = 8
EXAMPLE_SEED = 26_003


In [ ]:
with open(RUN_DIR / "config.json", "r") as f:
    cfg = json.load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
vocab = project_tokens.build_vocab(cfg["task"])
model = project_model.build_model(cfg=cfg, device=device)
checkpoint = project_checkpoints.load_checkpoint(path=CHECKPOINT_PATH, model=model, map_location=device)
model.eval()
final_layer = cfg["model"]["n_layers"] - 1
ACTIVATION_SITE = f"blocks.{final_layer}.hook_resid_pre"
seq_len = cfg["task"]["seq_len"]
print(f"Loaded checkpoint step {checkpoint['step']} on {device}")
print(f"Activation site: {ACTIVATION_SITE}")


## Collect activations and line-state variables

In [ ]:
def state_variables(token_rows, line_lengths):
    all_counts, all_lines = [], []
    for row in token_rows.detach().cpu().tolist():
        count = 0
        line_number = 0
        row_counts, row_lines = [], []
        for token_id in row:
            char_len = 0 if token_id == vocab.newline_id else project_data.extract_character_count(vocab.decode_token(token_id))
            row_counts.append(count if token_id == vocab.newline_id else count + char_len)
            row_lines.append(line_number)
            if token_id == vocab.newline_id:
                count = 0
                line_number += 1
            else:
                count += char_len
        all_counts.append(row_counts)
        all_lines.append(row_lines)
    return np.asarray(all_counts, dtype=np.float32), np.asarray(all_lines, dtype=np.float32)

@torch.no_grad()
def collect_examples(n_examples, seed):
    acts, counts, lines, widths = [], [], [], []
    collected = 0
    batch_number = 0
    while collected < n_examples:
        size = min(COLLECTION_BATCH_SIZE, n_examples - collected)
        batch = project_data.make_batch(size, vocab, cfg["task"], device,
                                        seed=seed + batch_number * COLLECTION_BATCH_SIZE)
        _, cache = model.run_with_cache(batch.tokens, prepend_bos=False,
                                        names_filter=lambda name: name == ACTIVATION_SITE)
        batch_counts, batch_lines = state_variables(batch.tokens, batch.line_lengths)
        acts.append(cache[ACTIVATION_SITE].detach().cpu())
        counts.append(batch_counts)
        lines.append(batch_lines)
        widths.append(batch.line_lengths.detach().cpu().numpy().astype(np.float32))
        collected += size
        batch_number += 1
    return (
        torch.cat(acts).numpy()[:n_examples],
        np.concatenate(counts)[:n_examples],
        np.concatenate(lines)[:n_examples],
        np.concatenate(widths)[:n_examples],
    )

X_train, counts_train, lines_train, widths_train = collect_examples(N_TRAIN_EXAMPLES, TRAIN_SEED)
X_eval, counts_eval, lines_eval, widths_eval = collect_examples(N_EVAL_EXAMPLES, EVAL_SEED)
d_model = X_train.shape[-1]
print("Train/eval shapes:", X_train.shape, X_eval.shape)


## Fit unconstrained and constrained models

In [ ]:
width_mean = widths_train.mean()
width_std = max(widths_train.std(), 1.0)
phase_spline = SplineTransformer(n_knots=N_SPLINE_KNOTS, degree=3, include_bias=False)
phase_spline.fit(np.linspace(0.0, 1.5, 500)[:, None])
n_phase = phase_spline.n_features_out_
line_mean = lines_train.mean()
line_std = max(lines_train.std(), 1.0)

def phase_features(counts, widths):
    phase = np.clip(counts / widths[:, None], 0.0, 1.5)
    return phase_spline.transform(phase.reshape(-1, 1))

def additive_features(counts, lines, widths):
    n_examples, n_positions = counts.shape
    phase = phase_features(counts, widths)
    line_scaled = ((lines - line_mean) / line_std).reshape(-1, 1)
    return phase, line_scaled

phase_train, line_train = additive_features(counts_train, lines_train, widths_train)
phase_eval, line_eval = additive_features(counts_eval, lines_eval, widths_eval)
X_train_flat = X_train.reshape(-1, d_model)
X_eval_flat = X_eval.reshape(-1, d_model)

# Unconstrained additive model: phase spline plus one line-number vector.
unconstrained = Ridge(alpha=RIDGE_ALPHA).fit(
    np.concatenate([phase_train, line_train], axis=1), X_train_flat
)

# Separate line offsets for a flexible comparison.
max_line = int(max(lines_train.max(), lines_eval.max()))
onehot_train = np.eye(max_line + 1, dtype=np.float32)[lines_train.astype(int).reshape(-1)]
onehot_eval = np.eye(max_line + 1, dtype=np.float32)[lines_eval.astype(int).reshape(-1)]
separate_offsets = Ridge(alpha=RIDGE_ALPHA).fit(
    np.concatenate([phase_train, onehot_train], axis=1), X_train_flat
)

# Alternating constrained fit. The phase coefficient vectors are projected
# into the orthogonal complement of the current line vector at each step.
phase_fit = Ridge(alpha=RIDGE_ALPHA, fit_intercept=False).fit(phase_train, X_train_flat)
phase_coef = phase_fit.coef_
line_fit = Ridge(alpha=RIDGE_ALPHA).fit(
    np.concatenate([phase_train, line_train], axis=1), X_train_flat
)
line_vector = line_fit.coef_[:, n_phase].copy()
intercept = line_fit.intercept_.copy()
for _ in range(N_ALTERNATING_STEPS):
    line_unit = line_vector / max(np.linalg.norm(line_vector), 1e-12)
    projection = np.eye(d_model) - np.outer(line_unit, line_unit)
    phase_coef = projection @ phase_coef
    phase_contribution = phase_train @ phase_coef.T
    residual = X_train_flat - phase_contribution
    line_design = np.column_stack([np.ones(len(line_train)), line_train[:, 0]])
    line_parameters = np.linalg.lstsq(line_design, residual, rcond=None)[0]
    intercept = line_parameters[0]
    line_vector = line_parameters[1]
    residual_without_line = X_train_flat - intercept - line_train[:, 0, None] * line_vector
    phase_fit = Ridge(alpha=RIDGE_ALPHA, fit_intercept=False).fit(phase_train, residual_without_line)
    phase_coef = projection @ phase_fit.coef_

print("Phase features:", n_phase)
print("Constrained line/phase dot product:", float(np.linalg.norm(phase_coef.T @ line_vector)))


## Held-out activation error and subspace diagnostics

In [ ]:
def constrained_predict(phase_matrix, line_matrix):
    return intercept + line_matrix[:, 0, None] * line_vector + phase_matrix @ phase_coef.T

predictions = {
    "unconstrained phase + line": unconstrained.predict(np.concatenate([phase_eval, line_eval], axis=1)).reshape(N_EVAL_EXAMPLES, seq_len, d_model),
    "separate line offsets": separate_offsets.predict(np.concatenate([phase_eval, onehot_eval], axis=1)).reshape(N_EVAL_EXAMPLES, seq_len, d_model),
    "orthogonal phase + line": constrained_predict(phase_eval, line_eval).reshape(N_EVAL_EXAMPLES, seq_len, d_model),
}
rows = []
mean_baseline = X_train_flat.mean(axis=0)
baseline_mse = np.mean((X_eval_flat - mean_baseline) ** 2)
for name, prediction in predictions.items():
    mse = float(np.mean((X_eval - prediction) ** 2))
    rows.append({
        "model": name,
        "activation_MSE": mse,
        "variance_reduction": 1.0 - mse / baseline_mse,
        "weighted_R2": r2_score(X_eval_flat, prediction.reshape(-1, d_model), multioutput="variance_weighted"),
    })
result_table = pd.DataFrame(rows)
display(result_table.style.format({
    "activation_MSE": "{:.6f}", "variance_reduction": "{:+.2%}", "weighted_R2": "{:.4f}"
}))


## Behavioral intervention: replace pre-attention activations

In [ ]:
@torch.no_grad()
def evaluate_replacement(replacement_activations=None):
    correct = total = true_positives = actual_newlines = predicted_newlines = 0
    for start in range(0, N_EVAL_EXAMPLES, COLLECTION_BATCH_SIZE):
        size = min(COLLECTION_BATCH_SIZE, N_EVAL_EXAMPLES - start)
        batch = project_data.make_batch(batch_size=size, vocab=vocab, task_cfg=cfg["task"], device=device, seed=EVAL_SEED + start)
        if replacement_activations is None:
            logits = model(batch.tokens)
        else:
            replacement = torch.as_tensor(replacement_activations[start:start + size], device=device, dtype=model.W_Q.dtype)
            def replace_activation(activation, hook, replacement=replacement):
                return replacement.to(device=activation.device, dtype=activation.dtype)
            logits = model.run_with_hooks(batch.tokens, prepend_bos=False, fwd_hooks=[(ACTIVATION_SITE, replace_activation)])
        predictions_here = logits.argmax(dim=-1)
        targets = batch.targets
        target_newline = targets == vocab.newline_id
        prediction_newline = predictions_here == vocab.newline_id
        correct += int((predictions_here == targets).sum().item())
        total += predictions_here.numel()
        true_positives += int((target_newline & prediction_newline).sum().item())
        actual_newlines += int(target_newline.sum().item())
        predicted_newlines += int(prediction_newline.sum().item())
    return {
        "overall_accuracy": correct / total,
        "newline_precision": true_positives / predicted_newlines if predicted_newlines else 0.0,
        "newline_recall": true_positives / actual_newlines if actual_newlines else np.nan,
        "newline_prediction_frequency": predicted_newlines / total,
    }
intervention_rows = [{"condition": "clean", **evaluate_replacement()}]
for name in ["unconstrained phase + line", "orthogonal phase + line"]:
    intervention_rows.append({"condition": name, **evaluate_replacement(predictions[name])})
intervention_results = pd.DataFrame(intervention_rows)
display(intervention_results.style.format({"overall_accuracy": "{:.4f}", "newline_precision": "{:.4f}", "newline_recall": "{:.4f}", "newline_prediction_frequency": "{:.4f}"}))


## Random 3D subspace baseline

In [ ]:
N_RANDOM_SUBSPACES = 20
RANDOM_SUBSPACE_SEED = 26_005
activation_mean = X_train_flat.mean(axis=0)
random_subspace_rows = []
random_subspace_reconstructions = {}
for trial in range(N_RANDOM_SUBSPACES):
    rng = np.random.default_rng(RANDOM_SUBSPACE_SEED + trial)
    random_basis, _ = np.linalg.qr(rng.standard_normal((d_model, 3)))
    centered = X_eval_flat - activation_mean
    reconstruction = activation_mean + (centered @ random_basis) @ random_basis.T
    reconstruction = reconstruction.reshape(N_EVAL_EXAMPLES, seq_len, d_model)
    random_subspace_reconstructions[trial] = reconstruction
    mse = float(np.mean((X_eval - reconstruction) ** 2))
    behavior = evaluate_replacement(reconstruction)
    random_subspace_rows.append({"trial": trial, "activation_MSE": mse, "variance_reduction": 1.0 - mse / baseline_mse, **behavior})
random_subspace_results = pd.DataFrame(random_subspace_rows)
random_subspace_summary = pd.DataFrame([{
    "condition": "random 3D subspace",
    **{column: random_subspace_results[column].mean() for column in ["activation_MSE", "variance_reduction", "overall_accuracy", "newline_precision", "newline_recall", "newline_prediction_frequency"]},
    **{f"{column}_std": random_subspace_results[column].std(ddof=1) for column in ["activation_MSE", "variance_reduction", "overall_accuracy", "newline_precision", "newline_recall", "newline_prediction_frequency"]},
}])
display(random_subspace_summary.style.format({"activation_MSE": "{:.6f}", "variance_reduction": "{:.2%}", "newline_precision": "{:.4f}", "newline_recall": "{:.4f}"}))
display(random_subspace_results.describe().T[["mean", "std", "min", "max"]].style.format("{:.6f}"))


## Orthogonality and phase-subspace rank

In [ ]:
line_unit = line_vector / max(np.linalg.norm(line_vector), 1e-12)
phase_grid = np.linspace(0.0, 1.5, 200)
phase_curve = phase_spline.transform(phase_grid[:, None]) @ phase_coef.T
phase_curve -= phase_curve.mean(axis=0, keepdims=True)
_, phase_singular_values, phase_vt = np.linalg.svd(phase_curve, full_matrices=False)
phase_explained = phase_singular_values ** 2 / np.sum(phase_singular_values ** 2)
phase_basis = phase_vt[:2].T
print("Phase singular-value fractions:", phase_explained[:5])
print(f"First-two phase variance: {phase_explained[:2].sum():.2%}")
print("Cosines between line vector and phase directions:", np.abs(phase_basis.T @ line_unit))


## Overlay a held-out trajectory

In [ ]:
example_index = 0
pca = PCA(n_components=3, svd_solver="randomized", random_state=26_004).fit(X_train_flat)
actual_coordinates = pca.transform(X_eval[example_index])
model_coordinates = {
    name: pca.transform(prediction[example_index]) for name, prediction in predictions.items()
}
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode="markers", marker=dict(size=8, color="black"), name="Origin"))
fig.add_trace(go.Scatter3d(x=actual_coordinates[:,0], y=actual_coordinates[:,1], z=actual_coordinates[:,2], mode="lines+markers", marker=dict(size=5, color=counts_eval[example_index], colorscale="Viridis", colorbar=dict(title="Character count")), line=dict(color="black", width=5), name="Actual"))
colors = ["royalblue", "crimson", "seagreen"]
for (name, coords), color in zip(model_coordinates.items(), colors):
    fig.add_trace(go.Scatter3d(x=coords[:,0], y=coords[:,1], z=coords[:,2], mode="lines", line=dict(color=color, width=3, dash="dash"), name=name))
fig.update_layout(title=f"Held-out example {example_index}: constrained versus unconstrained trajectories", scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"), height=800)
display(fig)
